# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs as defined in the Croissant schema.

In [ ]:
# List all record sets and their @id, name, and fields
record_sets = list(dataset.record_sets())
print("Available record sets:")
for rs in record_sets:
    print(f'- RecordSet @id: {rs.id} | name: {rs.name}')
    for field in rs.fields:
        print(f'    Field @id: {field.id} | name: {field.name}')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.


In [ ]:
# Extract data from each record set using their @id
# We'll dynamically extract all record sets
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f'DataFrame for RecordSet @id: {rs.id}')
    print(f'Columns: {df.columns.tolist()}')
    print(df.head(), '\n')

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing numeric fields, and grouping data.
All references use the entity `@id` as required.

In [ ]:
# Choose the main survey record set for analysis (replace with actual @id if different)
main_record_set = record_sets[0].id  # Using the first record set; adjust as necessary
main_df = dataframes[main_record_set]

# Identify available numeric field @ids
numeric_fields = []
for field in record_sets[0].fields:
    if field.data_type in ['schema:Integer', 'schema:Float', 'schema:Number']:
        numeric_fields.append(field.id)

print('Numeric fields in main record set:', numeric_fields)

# We'll use GAD-7 score as an example; find its @id
gad7_field_id = None
for field in record_sets[0].fields:
    if 'GAD-7' in field.name or 'gad7' in field.id.lower():
        gad7_field_id = field.id
        break
if gad7_field_id is None and numeric_fields:
    gad7_field_id = numeric_fields[0]

print(f'Using numeric field @id for EDA: {gad7_field_id}')

# Filter to records where GAD-7 > 10
threshold = 10
if gad7_field_id in main_df.columns:
    filtered_df = main_df[main_df[gad7_field_id] > threshold].copy()
    print(f'Filtered records with {gad7_field_id} > {threshold}:')
    print(filtered_df.head())

    # Normalize the GAD-7 score
    filtered_df[f"{gad7_field_id}_normalized"] = (
        filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()
    ) / filtered_df[gad7_field_id].std()
    print(f'Normalized {gad7_field_id} for filtered records:')
    print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

    # Group by education level (if available)
    # Find education field @id
    group_field_id = None
    for field in record_sets[0].fields:
        if 'education' in field.name.lower() or 'education' in field.id.lower():
            group_field_id = field.id
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[gad7_field_id].mean().reset_index()
        print(f'Grouped mean {gad7_field_id} by {group_field_id}:')
        print(grouped_df.head())
else:
    print(f'Field {gad7_field_id} not found in DataFrame columns.')

## 5. Visualization
Visualize distributions and relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of GAD-7 scores
if gad7_field_id in main_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[gad7_field_id].dropna(), bins=12, kde=True)
    plt.title(f'Distribution of GAD-7 scores (@id: {gad7_field_id})')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Frequency')
    plt.show()

    # If grouped_df exists, plot mean GAD-7 by education
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,6))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[gad7_field_id])
        plt.title(f'Mean GAD-7 scores by Education (@id: {group_field_id})')
        plt.xlabel('Education Level')
        plt.ylabel('Mean GAD-7 Score')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrates how to:
- Load a dataset described by a Croissant schema using `mlcroissant`.
- Reference and extract data by entity `@id`.
- Review available fields and record sets.
- Perform basic exploratory data analysis, filtering, normalization, and grouping.
- Visualize data distributions and relationships.

Key observations in the sample EDA include the distribution of GAD-7 scores and potential relationships between mental health indicators and education level, supporting mental health assessment and policy planning for Kilifi County, Kenya.